# Museum Classifier — Supervised: Decision Tree + AdaBoost

**Preprocessing pipeline:**
- **Pipeline A** — ResNet18 (pretrained CNN feature extractor) → 512-d vectors

**Steps:**
1. Image Preprocessing — ResNet18 features extracted
2. Modeling × 5 — five AdaBoost+DT configs
3. Report — metrics, ROC, CV F1, summary table

**Dataset structure on Google Drive:**
```
MyDrive/appliedAI/
├── training/
│   ├── museum-indoor/
│   └── museum-outdoor/
├── museum_validation/
│   ├── museum-indoor/
│   └── museum-outdoor/
└── test/
```

In [ ]:
# Install missing packages (run once)
!pip install -q scikit-image opencv-python-headless torch torchvision

## Environment Setup

Run this notebook **locally** or on **Google Colab** — the cell below detects the environment automatically.

- **Local**: uses `~/Documents/Prog/AppliedAI` as the base folder
- **Colab**: mounts Google Drive and uses `MyDrive/appliedAI` — upload your dataset there first

In [ ]:
import os

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/appliedAI')  # ← adjust Drive path if needed
else:
    BASE_DIR = Path.home() / 'Documents' / 'Prog' / 'AppliedAI'  # local path

print(f'Running in: {"Google Colab" if _in_colab() else "local"} environment')
print(f'BASE_DIR: {BASE_DIR}')

In [ ]:
import os, sys, time, warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, roc_curve, auc, ConfusionMatrixDisplay,
                             precision_score, recall_score)
from sklearn.preprocessing import StandardScaler

from skimage.feature import hog, local_binary_pattern
from skimage import color as skcolor
import cv2

%matplotlib inline
warnings.filterwarnings("ignore")
import joblib

## Configuration

Adjust `BASE_DIR` to point to your dataset folder inside Google Drive if needed.

In [ ]:
TRAIN_DIR  = BASE_DIR / 'training'
VAL_DIR    = BASE_DIR / 'museum_validation'
TEST_DIR   = BASE_DIR / 'test'
OUTPUT_DIR = BASE_DIR / 'outputs_supervised'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES      = ['museum-indoor', 'museum-outdoor']   # must match folder names
IMG_SIZE     = 224
BATCH_SIZE   = 32
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
RANDOM_STATE = 42

print(f'Training dir    : {TRAIN_DIR}')
print(f'Validation dir  : {VAL_DIR}')
print(f'Test dir        : {TEST_DIR}')
print(f'Device          : {DEVICE}')
CHECKPOINT_DIR = BASE_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Checkpoint file paths — delete a file to force recomputation of that step
CKPT_RESNET_TRAIN = CHECKPOINT_DIR / 'sup_resnet_train.npz'
CKPT_RESNET_VAL   = CHECKPOINT_DIR / 'sup_resnet_val.npz'
CKPT_MODELS_R     = CHECKPOINT_DIR / 'sup_models_resnet.joblib'

## Dataset Loader

In [ ]:
class MuseumDataset(Dataset):
    """Loads labeled images from a root folder containing one sub-folder per class."""
    def __init__(self, root: Path, classes: list, transform=None):
        self.samples, self.transform = [], transform
        for idx, cls in enumerate(classes):
            d = root / cls
            if not d.exists():
                print(f'[WARN] Folder not found: {d}'); continue
            for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp'):
                for p in d.glob(ext):
                    self.samples.append((p, idx))
        counts = {cls: sum(1 for _, l in self.samples if l == i)
                  for i, cls in enumerate(classes)}
        print(f'  {root.name}: {len(self.samples)} images — ' +
              ', '.join(f'{cls}={n}' for cls, n in counts.items()))

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

In [ ]:
print('Loading datasets...')
train_dataset = MuseumDataset(TRAIN_DIR, CLASSES)
val_dataset   = MuseumDataset(VAL_DIR,   CLASSES)

if len(train_dataset) == 0:
    raise RuntimeError('No training images found — check TRAIN_DIR.')
if len(val_dataset) == 0:
    raise RuntimeError('No validation images found — check VAL_DIR.')

## Step 1-A — Image Preprocessing: ResNet18 (512-d)

Pretrained ResNet18 with the classification head replaced by `nn.Identity()` acts as a frozen
feature extractor. Each image is resized to 224×224, normalized with ImageNet statistics,
and produces a 512-dimensional embedding. Run separately on training and validation sets.

In [ ]:
import os

def _load_npz(path):
    d = np.load(path)
    return d['X'], d['y']

def _save_npz(path, X, y):
    np.savez_compressed(path, X=X, y=y)
    print(f'  [CKPT] Saved → {path}')

def extract_resnet_features(dataset, ckpt_path=None):
    """
    Extract 512-d feature vectors using a frozen pretrained ResNet18.
    The classification head is replaced with nn.Identity(); the backbone
    runs in eval mode with torch.no_grad() — no backpropagation occurs.

    Note on data augmentation:
    Augmentation (flipping, jitter, rotation) only provides real benefit
    during fine-tuning, where each training epoch forces the model to learn
    invariances through gradient updates. Here the network is frozen —
    applying augmentation would only produce a slightly different 512-d
    vector for the same image, with no benefit for the downstream
    AdaBoost / Decision Tree classifier.

    Required preprocessing:
    - Resize 224x224: the spatial scale for which ImageNet filters were calibrated
    - Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]): input
      distribution expected by the pretrained weights

    Checkpoint: if ckpt_path is provided and exists, features are loaded
    from disk instead of being recomputed. Delete the file to force
    re-extraction.
    """
    if ckpt_path and ckpt_path.exists():
        print(f'  Loading from checkpoint: {ckpt_path.name}')
        X, y = _load_npz(ckpt_path)
        print(f'    shape={X.shape}')
        return X, y
    resnet_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    dataset.transform = resnet_transform
    backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    backbone.fc = nn.Identity()
    backbone.eval().to(DEVICE)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    feats_list, labels_list = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            feats_list.append(backbone(imgs.to(DEVICE)).cpu().numpy())
            labels_list.append(lbls.numpy() if isinstance(lbls, torch.Tensor)
                               else np.array(lbls))
    dataset.transform = None
    X = np.concatenate(feats_list)
    y = np.concatenate(labels_list)
    print(f'    shape={X.shape}')
    if ckpt_path:
        _save_npz(ckpt_path, X, y)
    return X, y

print('[STEP 1-A] Extracting ResNet18 features...')
print('  Training set:')
X_resnet_train, y_train = extract_resnet_features(train_dataset, CKPT_RESNET_TRAIN)
print('  Validation set:')
X_resnet_val,   y_val   = extract_resnet_features(val_dataset,   CKPT_RESNET_VAL)

## Step 2 — Modeling: Decision Tree baselines + AdaBoost (× 5)

**7 configurations total:**
- **2 plain DT baselines** (no boosting) — establish the performance floor and quantify the gain from AdaBoost
- **5 AdaBoost+DT variants** — varying `n_estimators`, `learning_rate`, `max_depth`, `criterion`, and `min_samples_leaf`

Each configuration is trained on the training set, cross-validated (5-fold), and evaluated on the validation set.

In [ ]:
# 7 configurations: 2 plain DT baselines + 5 AdaBoost+DT variants.
# Plain DT baselines allow direct comparison of boosting vs a single tree.
# AdaBoost configs vary: n_estimators, learning_rate, max_depth (≤2),
# criterion (gini vs entropy), and min_samples_leaf.
# max_depth capped at 2 for computational feasibility on Colab free tier.
ADABOOST_CONFIGS = {
    # ── Baselines: plain Decision Tree (no boosting) ──────────────────────────
    # Used to quantify the gain from AdaBoost over a single tree.

    # Baseline 1: single stump — matches depth of AdaBoost stump configs
    'DT_depth1_gini': DecisionTreeClassifier(
        max_depth=1, criterion='gini', random_state=RANDOM_STATE),

    # Baseline 2: depth-2 tree — matches depth of best AdaBoost configs
    'DT_depth2_gini': DecisionTreeClassifier(
        max_depth=2, criterion='gini', random_state=RANDOM_STATE),

    # ── AdaBoost + DT (ensemble with boosting) ────────────────────────────────

    # Config 1: minimal stump ensemble — gini, no leaf constraint
    'Stump_n10_gini': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=1, criterion='gini', min_samples_leaf=1),
        n_estimators=10, learning_rate=1.0, algorithm='SAMME',
        random_state=RANDOM_STATE),

    # Config 2: more stumps, entropy criterion — explores impurity measure effect
    'Stump_n50_entropy': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=1, criterion='entropy', min_samples_leaf=1),
        n_estimators=50, learning_rate=1.0, algorithm='SAMME',
        random_state=RANDOM_STATE),

    # Config 3: deeper base learner, gini — richer splits, moderate lr
    'Depth2_n50_gini': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=2, criterion='gini', min_samples_leaf=1),
        n_estimators=50, learning_rate=0.5, algorithm='SAMME',
        random_state=RANDOM_STATE),

    # Config 4: depth=2, entropy + leaf constraint — reduces overfitting
    'Depth2_n100_entropy_leaf5': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=2, criterion='entropy', min_samples_leaf=5),
        n_estimators=100, learning_rate=0.1, algorithm='SAMME',
        random_state=RANDOM_STATE),

    # Config 5: larger ensemble, conservative lr, gini, no leaf constraint
    'Depth2_n200_gini_leaf1_learn0.01': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=2, criterion='gini', min_samples_leaf=1),
        n_estimators=200, learning_rate=0.01, algorithm='SAMME',
        random_state=RANDOM_STATE),
}

In [ ]:
def run_pipeline(X_train: np.ndarray, y_train: np.ndarray,
                 X_val:   np.ndarray, y_val:   np.ndarray,
                 label: str, ckpt_path=None, n_splits: int = 5) -> tuple:
    """
    Scale → 5-fold CV on train → train → evaluate on val.
    Returns (results, scaler).

    Incremental checkpoint strategy:
    - Loads existing results from ckpt_path (if present)
    - Only trains configs not yet in the checkpoint
    - Merges and saves the combined results back to ckpt_path
    This allows adding new configs (e.g. DT baselines) without retraining
    existing AdaBoost models — previously saved weights are reused as-is.
    """
    results = {}
    scaler  = None

    # Load existing checkpoint (may be partial — only some configs saved)
    if ckpt_path and ckpt_path.exists():
        payload = joblib.load(ckpt_path)
        results = payload['results']
        scaler  = payload['scaler']
        print(f'\n[STEP 2] Checkpoint loaded: {ckpt_path.name}')
        for name, r in results.items():
            print(f'  [LOADED] {name}  Acc={r["acc"]:.4f}  F1={r["f1"]:.4f}  AUC={r["roc_auc"]:.4f}')

    # Identify configs missing from the checkpoint
    missing = {name: clf for name, clf in ADABOOST_CONFIGS.items()
               if name not in results}

    if not missing:
        print('  [CKPT] All configs already trained — nothing to do.')
        return results, scaler

    # StandardScaler applied for pipeline consistency.
    # Note: Decision Trees are scale-invariant (ordinal threshold splits) —
    # normalization does not affect results for DT or AdaBoost+DT.
    # Kept to ensure compatibility if a scale-sensitive model (SVM, kNN,
    # Logistic Regression) is added for comparison in future experiments.
    if scaler is None:
        scaler     = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
    else:
        X_train_sc = scaler.transform(X_train)
    X_val_sc = scaler.transform(X_val)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    print(f'\n{"-"*64}')
    print(f'  {label}')
    print(f'  features={X_train.shape[1]}  train={len(X_train)}  val={len(X_val)}')
    print(f'  Training {len(missing)} new config(s): {list(missing.keys())}')
    print(f'{"-"*64}')

    for name, clf in missing.items():
        print(f'  [MODEL] {name}')
        t0  = time.time()
        cv  = cross_val_score(clf, X_train_sc, y_train, cv=skf, scoring='f1', n_jobs=-1)
        clf.fit(X_train_sc, y_train)
        y_pred  = clf.predict(X_val_sc)
        y_proba = clf.predict_proba(X_val_sc)[:, 1]

        acc         = accuracy_score(y_val, y_pred)
        f1          = f1_score(y_val, y_pred)
        fpr, tpr, _ = roc_curve(y_val, y_proba)
        roc_auc     = auc(fpr, tpr)
        elapsed     = time.time() - t0

        results[name] = dict(
            clf=clf, scaler=scaler,
            acc=acc, f1=f1, roc_auc=roc_auc,
            cv_f1=cv, fpr=fpr, tpr=tpr,
            cm=confusion_matrix(y_val, y_pred),
            y_pred=y_pred, y_proba=y_proba, y_test=y_val,
            time=elapsed,
            report=classification_report(y_val, y_pred, target_names=CLASSES),
        )
        print(f'    Acc={acc:.4f}  F1={f1:.4f}  AUC={roc_auc:.4f}  '
              f'CV_F1={cv.mean():.4f}±{cv.std():.4f}  t={elapsed:.1f}s')

    # Save combined checkpoint (previously loaded + newly trained results)
    if ckpt_path:
        joblib.dump({'results': results, 'scaler': scaler}, ckpt_path)
        print(f'  [CKPT] Saved → {ckpt_path}')
    return results, scaler

In [ ]:
# ── PIPELINE A ─ Run (or load from checkpoint) ──────────────────────────────
print('=' * 64)
print('  Running Pipeline A — ResNet18')
print('=' * 64)
# 5-fold CV  |  512-d features  |  saves to CKPT_MODELS_R
res_r, sc_r = run_pipeline(
    X_resnet_train, y_train, X_resnet_val, y_val,
    'Pipeline A - ResNet18', CKPT_MODELS_R, n_splits=5
)
print('\n[Pipeline A done — checkpoint saved. You can stop here and resume later.]')


## AdaBoost Learning Curves

Plots validation F1 as a function of `n_estimators` for each AdaBoost configuration.
Uses `staged_predict` on already-trained models — **no retraining required**.
Plain DT baselines appear as horizontal reference lines to show the boosting gain.

In [ ]:
# ── AdaBoost learning curves (uses already-trained models, no retraining) ────

def plot_learning_curves(results, X_val_sc, y_val):
    """
    Plot validation F1 as a function of n_estimators for each AdaBoost config.
    Uses staged_predict — iterates over already-fitted estimators, no retraining.
    Plain DT baselines are shown as horizontal dashed reference lines.
    """
    fig, ax = plt.subplots(figsize=(13, 5))
    fig.patch.set_facecolor('#0d0f1a')
    ax.set_facecolor('#161929')
    ax.tick_params(colors='#ccc')
    ax.xaxis.label.set_color('#ccc')
    ax.yaxis.label.set_color('#ccc')
    for sp in ax.spines.values():
        sp.set_color('#2a2d3e')

    ada_palette = sns.color_palette('tab10', 5)
    ada_idx     = 0

    for name, r in results.items():
        clf = r['clf']

        # Plain DT: horizontal dashed line (staged_predict not available)
        if isinstance(clf, DecisionTreeClassifier):
            ax.axhline(r['f1'], linestyle='--', linewidth=1.5, alpha=0.75,
                       label=f'{name}  (F1={r["f1"]:.3f})', color='white')
            continue

        # AdaBoost: F1 at each boosting stage — no retraining, reads estimators_ list
        staged_f1 = [f1_score(y_val, y_p)
                     for y_p in clf.staged_predict(X_val_sc)]
        ax.plot(range(1, len(staged_f1) + 1), staged_f1,
                color=ada_palette[ada_idx], lw=1.8,
                label=f'{name}  (final F1={r["f1"]:.3f})')
        ada_idx += 1

    ax.set_xlabel('n_estimators')
    ax.set_ylabel('F1 (validation)')
    ax.set_title('AdaBoost Learning Curves\n'
                 '(dashed lines = plain DT baselines — boosting gain above these lines)',
                 color='white', fontsize=11, fontweight='bold')
    ax.legend(fontsize=7.5, facecolor='#0d0f1a', labelcolor='white',
              loc='lower right')
    plt.tight_layout()
    out_lc = OUTPUT_DIR / 'learning_curves.png'
    plt.savefig(out_lc, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f'[LEARNING CURVES] Saved → {out_lc}')

# Scale val features once using the fitted scaler from the pipeline
X_val_sc_lc = sc_r.transform(X_resnet_val)
plot_learning_curves(res_r, X_val_sc_lc, y_val)

In [ ]:
# ── COMBINE RESULTS ────────────────────────────────────────────────────────
all_results = {'resnet': res_r}
scalers     = {'resnet': sc_r}
print('Results ready — generating report.')
for mn, r in all_results['resnet'].items():
    print(f'  {mn:35s}  Acc={r["acc"]:.4f}  F1={r["f1"]:.4f}  AUC={r["roc_auc"]:.4f}')

## Step 3 — Report

Generates a comprehensive figure with grouped bar charts, ROC curves, confusion matrices,
CV F1 heatmap, ΔF1 comparison, and a full summary table.

In [ ]:
PIPE_META = {
    'resnet': {'label': 'Pipeline A — ResNet18 (512-d)', 'color': '#4FC3F7'},
}

def build_report(all_results: dict):
    model_names = list(ADABOOST_CONFIGS.keys())
    n           = len(model_names)
    palette     = sns.color_palette('tab10', n)

    fig = plt.figure(figsize=(20, 34))
    fig.patch.set_facecolor('#0d0f1a')
    gs  = gridspec.GridSpec(5, 3, figure=fig, hspace=0.62, wspace=0.40)
    tkw  = dict(color='white', fontsize=10, fontweight='bold', pad=8)
    axbg = '#161929'
    color = PIPE_META['resnet']['color']

    def sa(ax):
        ax.set_facecolor(axbg); ax.tick_params(colors='#ccc')
        ax.xaxis.label.set_color('#ccc'); ax.yaxis.label.set_color('#ccc')
        for sp in ax.spines.values(): sp.set_color('#2a2d3e')

    xlbls = [m.replace('_', '\n') for m in model_names]
    x, w  = np.arange(n), 0.6

    # Row 0: Accuracy / F1 / AUC bars
    for col, (mkey, mtitle) in enumerate([('acc','Accuracy'),('f1','F1 Score'),('roc_auc','ROC-AUC')]):
        ax = fig.add_subplot(gs[0, col]); sa(ax)
        vals = [all_results['resnet'][m][mkey] for m in model_names]
        bars = ax.bar(x, vals, w, color=color, edgecolor='white', linewidth=0.4, alpha=0.88)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, v+0.012,
                    f'{v:.2f}', ha='center', color='white', fontsize=6.5)
        ax.set_xticks(x); ax.set_xticklabels(xlbls, color='#ccc', fontsize=6.5)
        ax.set_ylim(0, 1.15); ax.set_title(mtitle, **tkw)

    # Row 1: ROC curves (spans 2 cols) + CV F1 heatmap
    ax_roc = fig.add_subplot(gs[1, 0:2]); sa(ax_roc)
    ax_roc.plot([0,1],[0,1],'w--',lw=1,alpha=0.35,label='Random')
    for i, mn in enumerate(model_names):
        r = all_results['resnet'][mn]
        ax_roc.plot(r['fpr'], r['tpr'], color=palette[i], lw=1.8,
                label=f'{mn} ({r["roc_auc"]:.3f})')
    ax_roc.set_xlabel('False Positive Rate'); ax_roc.set_ylabel('True Positive Rate')
    ax_roc.set_title('ROC Curves — Pipeline A: ResNet18', **tkw)
    ax_roc.legend(fontsize=5.5, facecolor='#0d0f1a', labelcolor='white')

    ax_heat = fig.add_subplot(gs[1, 2]); sa(ax_heat)
    heat = np.array([[all_results['resnet'][mn]['cv_f1'].mean() for mn in model_names]])
    im = ax_heat.imshow(heat, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
    ax_heat.set_xticks(range(n)); ax_heat.set_xticklabels(xlbls, color='#ccc', fontsize=6.5)
    ax_heat.set_yticks([0]); ax_heat.set_yticklabels(['ResNet18'], color='#ccc', fontsize=8)
    ax_heat.set_title('CV F1 Mean Heatmap\n(5-fold on train, darker = better)', **tkw)
    for j in range(n):
        ax_heat.text(j, 0, f'{heat[0,j]:.3f}', ha='center', va='center',
                     color='black', fontsize=9, fontweight='bold')
    plt.colorbar(im, ax=ax_heat, fraction=0.046, pad=0.04)

    # Row 2: Best confusion matrix + CV F1 boxplot
    best_mn = max(model_names, key=lambda m: all_results['resnet'][m]['f1'])
    ax_cm = fig.add_subplot(gs[2, 0]); sa(ax_cm)
    ConfusionMatrixDisplay(all_results['resnet'][best_mn]['cm'],
                           display_labels=CLASSES).plot(ax=ax_cm, colorbar=False, cmap='Blues')
    ax_cm.set_title(f'Best CM — ResNet18\n{best_mn}', **tkw)
    ax_cm.xaxis.label.set_color('#ccc'); ax_cm.yaxis.label.set_color('#ccc')

    ax_box = fig.add_subplot(gs[2, 1:]); sa(ax_box)
    cv_data = [all_results['resnet'][m]['cv_f1'] for m in model_names]
    bp = ax_box.boxplot(cv_data, patch_artist=True,
                    medianprops=dict(color='white', linewidth=2))
    for patch, c in zip(bp['boxes'], palette):
        patch.set_facecolor(c); patch.set_alpha(0.75)
    ax_box.set_xticks(range(1, n+1)); ax_box.set_xticklabels(xlbls, color='#ccc', fontsize=6.5)
    ax_box.set_ylim(0, 1.1)
    ax_box.set_title('CV F1 Distribution — Pipeline A: ResNet18', **tkw)

    # Row 3: Per-class Precision & Recall
    class_colors = ['#4FC3F7', '#F48FB1']
    w2 = 0.35

    ax_prec = fig.add_subplot(gs[3, 0:2]); sa(ax_prec)
    ax_rec  = fig.add_subplot(gs[3, 2]);   sa(ax_rec)

    for ci, cls_name in enumerate(CLASSES):
        prec_vals = [precision_score(all_results['resnet'][m]['y_test'],
                                     all_results['resnet'][m]['y_pred'],
                                     average=None)[ci] for m in model_names]
        rec_vals  = [recall_score(all_results['resnet'][m]['y_test'],
                                   all_results['resnet'][m]['y_pred'],
                                   average=None)[ci] for m in model_names]
        offset = (ci - 0.5) * w2
        bars_p = ax_prec.bar(x + offset, prec_vals, w2, label=cls_name,
                             color=class_colors[ci], edgecolor='white', linewidth=0.4, alpha=0.88)
        bars_r = ax_rec.bar(x + offset, rec_vals, w2, label=cls_name,
                            color=class_colors[ci], edgecolor='white', linewidth=0.4, alpha=0.88)
        for bar, v in zip(bars_p, prec_vals):
            ax_prec.text(bar.get_x()+bar.get_width()/2, v+0.012,
                         f'{v:.2f}', ha='center', color='white', fontsize=6)
        for bar, v in zip(bars_r, rec_vals):
            ax_rec.text(bar.get_x()+bar.get_width()/2, v+0.012,
                        f'{v:.2f}', ha='center', color='white', fontsize=6)

    ax_prec.set_xticks(x); ax_prec.set_xticklabels(xlbls, color='#ccc', fontsize=6.5)
    ax_prec.set_ylim(0, 1.25)
    ax_prec.set_title('Precision per Class — Pipeline A: ResNet18', **tkw)
    ax_prec.legend(fontsize=7, facecolor='#0d0f1a', labelcolor='white')

    ax_rec.set_xticks(x); ax_rec.set_xticklabels(xlbls, color='#ccc', fontsize=6.5)
    ax_rec.set_ylim(0, 1.25)
    ax_rec.set_title('Recall per Class\n— Pipeline A: ResNet18', **tkw)
    ax_rec.legend(fontsize=7, facecolor='#0d0f1a', labelcolor='white')

    # Row 4: Training time + summary table
    ax_time = fig.add_subplot(gs[4, 0]); sa(ax_time)
    times = [all_results['resnet'][m]['time'] for m in model_names]
    ax_time.bar(x, times, w, color=color, edgecolor='white', linewidth=0.4, alpha=0.88)
    ax_time.set_xticks(x); ax_time.set_xticklabels(xlbls, color='#ccc', fontsize=6.5)
    ax_time.set_ylabel('seconds'); ax_time.set_title('Training Time (s)', **tkw)

    ax_tbl = fig.add_subplot(gs[4, 1:]); sa(ax_tbl); ax_tbl.axis('off')
    col_labels = ['Model','Accuracy','F1','ROC-AUC','CV F1 μ','CV F1 σ','Time (s)']
    rows = []
    for mn in model_names:
        r = all_results['resnet'][mn]
        rows.append([
            mn, f'{r["acc"]:.4f}', f'{r["f1"]:.4f}', f'{r["roc_auc"]:.4f}',
            f'{r["cv_f1"].mean():.4f}', f'{r["cv_f1"].std():.4f}', f'{r["time"]:.1f}',
        ])
    tbl = ax_tbl.table(cellText=rows, colLabels=col_labels, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.90)
    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor('#2a2d3e')
        if row == 0:
            cell.set_facecolor('#1e2235'); cell.set_text_props(color='white', fontweight='bold')
        else:
            cell.set_facecolor('#0d1828' if row%2 else '#091220')
            cell.set_text_props(color='#B3E5FC')
    ax_tbl.set_title('Results Summary — Pipeline A: ResNet18', **tkw)

    fig.text(0.5, 0.987, 'Museum Classifier — Supervised Decision Tree + AdaBoost',
             ha='center', va='top', color='white', fontsize=17, fontweight='bold')
    fig.text(0.5, 0.974,
             'Step 3 Report  |  Pipeline A: ResNet18 (512-d)',
             ha='center', va='top', color='#aaa', fontsize=10)

    out = OUTPUT_DIR / 'report_supervised.png'
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f'\n[REPORT] Saved → {out}')
    return out

In [ ]:
report_path = build_report(all_results)

### Note: CV F1 vs Validation F1

For all configs, the validation F1 (evaluated on `museum_validation/`) is slightly
higher than CV F1μ (5-fold cross-validation on `training/`). Three factors explain this:

1. **Statistical variance of the small validation set** — with n≈200 images in
   `museum_validation/`, the 95% confidence interval on F1 is ≈ ±0.017. The observed
   gap (≈0.02–0.03 pp) falls within normal estimation noise.

2. **CV is conservative by design** — each fold trains on 80% of the training data.
   The final model uses 100%, making it naturally slightly stronger and raising the
   validation F1.

3. **Independent distributions** — `training/` and `museum_validation/` are independent
   splits of the Places MIT dataset. Differences in image difficulty between sets are expected.

No evidence of data leakage: the gap shrinks for more complex configs (Depth2\_n200),
which is the opposite of what overfitting would produce.

In [ ]:
from scipy import stats

def f1_confidence_interval(f1, n, alpha=0.05):
    """Approximate 95% CI for F1 score using normal approximation."""
    z      = stats.norm.ppf(1 - alpha / 2)
    margin = z * np.sqrt(f1 * (1 - f1) / n)
    return margin

n_val = len(y_val)
print(f'Validation F1 with 95% confidence intervals  (n={n_val}):')
print(f'{"Model":<38}  {"F1":>6}  {"±95% CI":>8}  {"CV F1μ":>8}  {"gap":>6}')
print('-' * 72)
for name, r in res_r.items():
    margin = f1_confidence_interval(r['f1'], n_val)
    gap    = r['f1'] - r['cv_f1'].mean()
    print(f'  {name:<36}  {r["f1"]:.4f}  ±{margin:.4f}  '
          f'{r["cv_f1"].mean():.4f}   {gap:+.4f}')

## Best Model Classification Reports

In [ ]:
for pk in all_results:
    best = max(all_results[pk], key=lambda m: all_results[pk][m]['f1'])
    print(f'[BEST — {PIPE_META[pk]["label"]}]  {best}')
    print(all_results[pk][best]['report'])
    print()

## Test Prediction

Uses the best model (Pipeline A — ResNet18) to predict unlabeled images in `TEST_DIR`.
Saves a CSV with filename, predicted class, confidence, pipeline, and model name.


In [ ]:
def predict_test(all_results: dict, scalers: dict):
    best_pk, best_mn, best_f1 = None, None, -1
    for pk in all_results:
        for mn in all_results[pk]:
            if all_results[pk][mn]['f1'] > best_f1:
                best_pk, best_mn, best_f1 = pk, mn, all_results[pk][mn]['f1']

    print(f'Best model: {PIPE_META[best_pk]["label"]} / {best_mn}  (F1={best_f1:.4f})')

    test_paths = []
    if TEST_DIR.exists():
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp'):
            test_paths.extend(TEST_DIR.glob(ext))
    if not test_paths:
        print('[WARN] No test images found in TEST_DIR.'); return

    clf, scaler = all_results[best_pk][best_mn]['clf'], scalers[best_pk]

    backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    backbone.fc = nn.Identity(); backbone.eval().to(DEVICE)
    tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    feats = []
    with torch.no_grad():
        for p in test_paths:
            img = tf(Image.open(p).convert('RGB')).unsqueeze(0).to(DEVICE)
            feats.append(backbone(img).cpu().numpy()[0])

    X_test = scaler.transform(np.array(feats))
    preds  = clf.predict(X_test)
    probas = clf.predict_proba(X_test)[:, 1]

    out_csv = OUTPUT_DIR / 'predictions_supervised.csv'
    with open(out_csv, 'w') as f:
        f.write('filename,prediction,confidence_outdoor,pipeline,model\n')
        for p, pred, prob in zip(test_paths, preds, probas):
            f.write(f'{p.name},{CLASSES[pred]},{prob:.4f},{best_pk},{best_mn}\n')
    print(f'Predictions saved \u2192 {out_csv}')
    print(f'[DONE] All outputs in: {OUTPUT_DIR}')

predict_test(all_results, scalers)
